# overall_sleep: records, coverage and defensible relationships
For Friday October 23, 2026. Only this table and the small/aggregated comparisons needed below are read, inside a read-only repeatable-read snapshot. No database changes are made. Outputs contain aggregate statistics and anonymous plots, not participant identifiers or raw records.

Shared IDs/group/type identify context; calendar/week/day label the export/reporting period; start/end/duration describe its window. Score definitions, label rules, timestamp conventions and device algorithms are unverified without the source dictionary. Names and SQL types are not definitions. No clinical thresholds, diagnosis, causation or predictive validation are used.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display, JSON, Markdown
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately in .env.'
engine=create_engine(url); del url
connection=engine.connect().execution_options(isolation_level='REPEATABLE READ')
transaction=connection.begin()
connection.execute(text('SET TRANSACTION READ ONLY'))
connection.execute(text("SET LOCAL statement_timeout = '240s'"))
def query(sql): return pd.read_sql(text(sql),connection)
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
results={}
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
def show_schema(table):
    cols=inspect(connection).get_columns(table)
    display(pd.DataFrame([{'column':c['name'],'SQL_type':str(c['type'])} for c in cols]))
    return [c['name'] for c in cols]
def audit_keys(df,candidates):
    display(pd.DataFrame([{'key':name,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
      'rows_in_duplicate_keys':int(df.duplicated(k,keep=False).sum()),'unique_nonnull':valid_key(df,k)} for name,k in candidates.items()]))
def period_audit(p):
    p=p.copy(); p['calendar_date']=pd.to_datetime(p.calendar_date,errors='coerce')
    elapsed=p.end_measure_period_sec-p.start_measure_period_sec
    ordered=p.sort_values(key)
    prior=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
    utc=pd.to_datetime(p.start_measure_period_sec,unit='s',utc=True,errors='coerce')
    stats={'periods':len(p),'participants':p.sdv_user_id.nunique(),
      'null_key_rows':int(p[key].isna().any(axis=1).sum()),'duplicate_period_keys':int(p.duplicated(key,keep=False).sum()),
      'nonpositive_periods':int(elapsed.le(0).sum()),'exactly_24_hour_periods':int(elapsed.eq(86400).sum()),
      'overlapping_periods':int(ordered.start_measure_period_sec.lt(prior).sum()),
      'UTC_start_date_matches_label':int(utc.dt.tz_localize(None).dt.normalize().eq(p.calendar_date).sum())}
    if 'duration_measure_period_sec' in p:
        stats['duration_equals_end_minus_start']=int(p.duration_measure_period_sec.eq(elapsed).sum())
    display(pd.DataFrame([stats])); display((elapsed/3600).value_counts().sort_index().rename_axis('period_hours').to_frame('periods'))
    coverage=p.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),observed_dates=('calendar_date','nunique'),periods=('calendar_date','size'))
    coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
    coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
    coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
    display(coverage[['observed_dates','periods','span_days','missing_dates_within_span','observed_date_percent']].describe().T)
    stats.update({'min_observed_dates_per_participant':int(coverage.observed_dates.min()),
      'max_observed_dates_per_participant':int(coverage.observed_dates.max()),
      'min_observed_date_percent':coverage.observed_date_percent.min(),'max_observed_date_percent':coverage.observed_date_percent.max()})
    return stats
def checked_join(left,right,label):
    if not (valid_key(left,key) and valid_key(right,key)):
        print(label+': skipped; keys are not unique and non-null.'); return None
    joined=left.merge(right,on=key,how='outer',suffixes=('_left','_right'),indicator=True,validate='one_to_one')
    counts={str(k):int(v) for k,v in joined._merge.value_counts().items()}
    results[label+'_period_join']=counts
    results[label+'_participant_overlap']={'left':left.sdv_user_id.nunique(),'right':right.sdv_user_id.nunique(),
      'shared':len(set(left.sdv_user_id.dropna()) & set(right.sdv_user_id.dropna()))}
    display(pd.Series(counts,name='periods').rename_axis(label+'_status').to_frame())
    return joined

TABLE='overall_sleep'
columns=show_schema(TABLE)
d=query('SELECT * FROM '+TABLE)
d['calendar_date']=pd.to_datetime(d.calendar_date,errors='coerce')
results['inventory']={'rows':len(d),'participants':d.sdv_user_id.nunique(),'null_cells':int(d.isna().sum().sum()),'exact_duplicate_rows_after_first':int(d.duplicated().sum())}
display(pd.DataFrame([results['inventory']]))
display(pd.DataFrame({'column':d.columns,'pandas_type':d.dtypes.astype(str).values,'missing_rows':d.isna().sum().values,'distinct_nonnull':d.nunique().values}))


## 1. Summary grain, keys and date coverage
Test record ID and participant/date/period keys before treating a row as one summary. A date label does not prove a complete night or full wear. Missing summaries remain absent, not zero. Inspect exact duplicates and nulls separately from operational sentinel/label checks.

In [ ]:
audit_keys(d,{'record_id':['additional_id'],'participant_date':['sdv_user_id','calendar_date'],'period':key})
results['periods']=period_audit(d)


## Reporting-window interpretation
Reporting windows are checked separately from actual observations. End minus start, nonpositive windows, exact 24-hour windows and overlaps are diagnostics, not proof of wear or complete days. UTC conversion assumes Unix seconds only as a plausibility check; local calendar labels and inclusive/exclusive endpoints need documentation. Date coverage is unique observed dates divided by dates in each participant's first-to-last observed span; enrolment boundaries are unknown. Missing dates are never zero values. Participant identifiers are retained only in memory for grouping and are not displayed.

## 2. Recorded sleep score and qualifier
`sleep_value` is a source score with unverified scale/algorithm; `sleep_qualifier` is a recorded label, not a clinical diagnosis. Describe null/zero/negative values and the score distribution by qualifier. No threshold mapping or recommended range is inferred. More-observed participants contribute more records; participant medians are summarized without identifiers.

In [ ]:
x=d.sleep_value
results['score_quality']={'missing':int(x.isna().sum()),'zero':int(x.eq(0).sum()),'negative':int(x.lt(0).sum()),'minimum':x.min(),'median':x.median(),'maximum':x.max()}
results['qualifiers']={str(k):int(v) for k,v in d.sleep_qualifier.fillna('<missing>').value_counts().items()}
display(pd.DataFrame([results['score_quality']]))
display(d.sleep_qualifier.fillna('<missing>').value_counts().rename_axis('qualifier').to_frame('records'))
display(d.groupby('sleep_qualifier',dropna=False).sleep_value.agg(['count','median','min','max']))
display(d.groupby('sdv_user_id').sleep_value.median().describe().to_frame('distribution_of_participant_medians'))
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
x.dropna().plot.hist(bins=20,ax=axes[0]); axes[0].set(title='Recorded sleep score',xlabel='Score; source scale unverified',ylabel='Records')
d.sleep_qualifier.fillna('<missing>').value_counts().plot.barh(ax=axes[1]); axes[1].set(title='Recorded sleep qualifiers',xlabel='Records')
plt.tight_layout(); plt.show()


## 3. Validated relationships: summary pairing and event coverage
Compare the two summary tables on participant/date/start/end with unique/non-null keys and explicit one-to-one cardinality. Check shared ID/group/duration. For events, aggregate to one row per exact period first, retaining context-variation counts; never merge every event directly with repeated summaries.

The event table may cover more participants and periods. Unmatched counts quantify coverage differences rather than zero scores or no events. Calendar dates may label a reporting day, not the night-start date. This cell counts event rows/periods only; the dedicated event notebook handles interval unions, cross-midnight and overlap.

In [ ]:
other=query('SELECT * FROM sleep'); other['calendar_date']=pd.to_datetime(other.calendar_date,errors='coerce')
audit_keys(other,{'period':key,'participant_date':['sdv_user_id','calendar_date']})
j=checked_join(d,other,'sleep')
if j is not None:
    both=j._merge.eq('both')
    results['summary_context']={field+'_equal':int((both & j[field+'_left'].eq(j[field+'_right'])).sum()) for field in ['additional_id','sdv_group_id','duration_measure_period_sec']}
    display(pd.crosstab(j.loc[both,'total_duration'],j.loc[both,'sleep_qualifier']))
    display(j.loc[both].groupby('total_duration').sleep_value.agg(['count','median','min','max']))
events=query("""SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,
 COUNT(*) event_rows,COUNT(DISTINCT additional_id) record_variants,MIN(additional_id) additional_id,
 COUNT(DISTINCT sdv_group_id) group_variants,MIN(sdv_group_id) sdv_group_id,
 COUNT(DISTINCT duration_measure_period_sec) duration_variants,MIN(duration_measure_period_sec) duration_measure_period_sec
 FROM sleep_level_events GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec""")
events['calendar_date']=pd.to_datetime(events.calendar_date,errors='coerce')
ej=checked_join(d,events,'sleep_level_events')
if ej is not None:
    both=ej._merge.eq('both'); unique=both & ej.record_variants.eq(1) & ej.group_variants.eq(1) & ej.duration_variants.eq(1)
    results['event_context']={field+'_equal':int((unique & ej[field+'_left'].eq(ej[field+'_right'])).sum()) for field in ['additional_id','sdv_group_id','duration_measure_period_sec']}
results['event_coverage']={'events':int(events.event_rows.sum()),'periods':len(events),'participants':events.sdv_user_id.nunique()}


## Verified findings and teacher takeaway

- **299 records, five participants**, no null cells or exact duplicates. Tested record/participant-date/period keys are unique and non-null. Recorded scores have no zero/negative values and range **26–98**, median **76**; source scale and algorithm remain unverified.
- Source qualifiers are **FAIR 126**, **GOOD 92**, **POOR 60**, **EXCELLENT 21**. These are recorded categories, not independent medical classifications or thresholds inferred here.
- **293 reporting windows are exactly 24 hours**, six are not. Duration equals end minus start in all records; no reporting-window overlap is detected. Participants have **11–121 observed dates** and **78.57%–99.18% observed-date coverage** within first-to-last spans, excluding unknown enrolment/wear time.
- Exact-period join with `sleep` gives **299 one-to-one matches, no unmatched periods**, and agreeing IDs/groups/durations. Both tables cover the same five participants.
- Sleep-level events contain **18,402 intervals across 797 periods/eight participants**. All **299 overall summaries match event aggregates** with agreeing context; **498 event periods lack an overall-summary match**. All five summary participants occur in events. Missing summaries remain absent, not zero sleep or a zero score.

**Takeaway:** “Overall sleep is a source score/qualifier table for a smaller cohort and fewer periods than the event export. I validated exact period matches and coverage before comparing it with component labels. The score scale, label rules and complete-night/wear interpretation still need documentation.”


## Teacher-facing method and unresolved meanings
“I established whether rows were summaries or events, checked missing/duplicate/invalidity candidates, and validated period keys before any comparison. I kept unequal participant coverage and unmatched records visible.”

For events, sums can double-count overlapping intervals; interval unions use an explicit diagnostic convention. For sleep strings, labels are not numeric durations or percentages. Source score/impact/label definitions, algorithms, endpoints, timezone, enrolment and actual wear remain unresolved. Repeated records and unequal follow-up limit pooled interpretation. No medical, causal or predictive conclusion is made. CSV export, RAG and model building remain deferred.

In [ ]:
# End the read-only snapshot. No database changes were made.
transaction.rollback(); connection.close(); engine.dispose()
def scalar(v):
    if hasattr(v,'item'): return v.item()
    return str(v)
safe_results=json.loads(json.dumps(results,default=scalar))
display(JSON(safe_results))
print('All analysis cells completed; read-only connection closed.')
